In [ ]:
import time
import json
import os
import sys
import math
sys.path.append(os.path.dirname(os.getcwd()))
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import ADC,DAC,CHIP
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import plot_v_cond,plot_cond,show_crossbar,DataLoader

from network.layer import Layer

from scipy.stats import norm
from scipy.optimize import curve_fit

In [ ]:
chip=CHIP(PS(host="192.168.1.10", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0,IsNew32=False)
chip.adc.set_gap(adc_cs_gap=90,adc_first_gap=100,adc_last_gap=100)
chip.adc.set_gain_resistor(big_resistance=10e3,small_resistance=200)
chip.clk_manager.set_cyc(10, 10,delay3=50)
chip.add_compiler("../compiler/code/")
chip.compensation.initop("../chip_data/chip6_/")

# 1. 准备数据

In [ ]:
v,c_expected_from_row,r = chip.read4(crossbar=np.ones((256,256)),row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
c_expected_from_row = chip.compensation.compensation_point(r,from_row=True,return_type=0)

v,c_expected_from_col,r = chip.read4(crossbar=np.ones((256,256)),row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=False,split_type=0,row_type=0,col_type=0)
c_expected_from_col = chip.compensation.compensation_point(r,from_row=False,return_type=0)

## 1.1 从行给信号

In [ ]:
chip.compensation.initop("../chip_data/chip6_/")
# 10列，两路TIA并行
col_index = [i for i in range(256)]
# 存储数据
real_res = np.zeros((256,256))
real_cond_res = np.zeros((256,256))
expected_res = np.zeros((256,256))

for i in range(101):
    row_index = [j for j in range(i+1)]

    _,cond,r = chip.read4(crossbar=np.ones((256,256)),row_index=row_index,col_index=col_index,read_voltage=0.1,tg=5,gain=3,sub_base=True,from_row=True,split_type=3,row_type=0,col_type=0)
    cond = chip.compensation.compensation_forward(row_index,r,from_row=True,return_type=0,parallel=0)
    pos = np.ix_(row_index,col_index)
    expected_res[i,:]=np.sum(c_expected_from_row[pos],axis=0)
    real_res[i,:]=r
    real_cond_res[i,:] = cond[col_index]

In [ ]:
startpos,endpos = 0,101
for num in range(10):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(real_cond_res[startpos:endpos,num].flatten(),label = "actual output")
    plt.plot(expected_res[startpos:endpos,num].flatten(),label = "expected output")
    plt.title(f"col={col_index[num]}")
    plt.ylabel("uS")
    plt.xlabel("row nums")
    plt.legend()
    # plt.show()
    plt.subplot(1,2,2)
    plt.plot(real_cond_res[startpos:endpos,num].flatten()/expected_res[startpos:endpos,num].flatten(),label = "actual output/expected output")
    plt.plot(np.ones(endpos-startpos-1),label = "expected output/expected output")
    plt.title(f"col={col_index[num]}")
    plt.ylabel("*100%")
    plt.xlabel("row nums")
    plt.legend()
    plt.show()

In [ ]:
chip.compensation.initop("../chip_data/chip6_/")
# 10列，两路TIA并行
col_index = [i for i in range(256)]
# 存储数据
real_res = np.zeros((256,256))
real_cond_res = np.zeros((256,256))
expected_res = np.zeros((256,256))

for i in range(101):
    row_index = [j for j in range(i+1)]

    _,cond,r = chip.read4(crossbar=np.ones((256,256)),row_index=row_index,col_index=col_index,read_voltage=0.1,tg=5,gain=3,sub_base=True,from_row=True,split_type=4,row_type=0,col_type=0)
    cond = chip.compensation.compensation_forward(row_index,r,from_row=True,return_type=0,parallel=16)
    pos = np.ix_(row_index,col_index)
    expected_res[i,:]=np.sum(c_expected_from_row[pos],axis=0)
    real_res[i,:]=r
    real_cond_res[i,:] = cond[col_index]

In [ ]:
startpos,endpos = 0,101
for num in range(10):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(real_cond_res[startpos:endpos,num].flatten(),label = "actual output")
    plt.plot(expected_res[startpos:endpos,num].flatten(),label = "expected output")
    plt.title(f"col={col_index[num]}")
    plt.ylabel("uS")
    plt.xlabel("row nums")
    plt.legend()
    # plt.show()
    plt.subplot(1,2,2)
    plt.plot(real_cond_res[startpos:endpos,num].flatten()/expected_res[startpos:endpos,num].flatten(),label = "actual output/expected output")
    plt.plot(np.ones(endpos-startpos-1),label = "expected output/expected output")
    plt.title(f"col={col_index[num]}")
    plt.ylabel("*100%")
    plt.xlabel("row nums")
    plt.legend()
    plt.show()

## 1.2 从列给信号

In [ ]:
# offset = self.col_offset if from_row else self.row_offset
# # value
# value = self.col_value if from_row else self.row_value
chip.compensation.row_offset=0
chip.compensation.row_value=0

In [ ]:
chip.compensation.initop("../chip_data/chip6_/")
# chip.compensation.row_offset=0
# chip.compensation.row_value=0
# chip.compensation.row_r_out=0
# 10列，两路TIA并行
row_index = [i for i in range(256)]
# 存储数据
real_res = np.zeros((256,256))
real_cond_res = np.zeros((256,256))
expected_res = np.zeros((256,256))
chip.setting.tia_map = [j for j in range(8) for i in range(2)]
for i in range(101):
    col_index = [j for j in range(i+1)]

    _,cond,r = chip.read4(crossbar=np.ones((256,256)),row_index=row_index,col_index=col_index,read_voltage=0.1,tg=5,gain=3,sub_base=True,from_row=False,split_type=3,row_type=0,col_type=0)
    cond = chip.compensation.compensation_forward(col_index,r,from_row=False,return_type=0)
    pos = np.ix_(row_index,col_index)
    expected_res[i,:]=np.sum(c_expected_from_col[pos],axis=1).flatten()
    real_res[i,:]=r
    real_cond_res[i,:] = cond[row_index]

In [ ]:
startpos,endpos = 0,101
for num in range(75):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(real_cond_res[startpos:endpos,num].flatten(),label = "actual output")
    plt.plot(expected_res[startpos:endpos,num].flatten(),label = "expected output")
    plt.title(f"row={row_index[num]}")
    plt.ylabel("uS")
    plt.xlabel("row nums")
    plt.legend()
    # plt.show()
    plt.subplot(1,2,2)
    plt.plot(real_cond_res[startpos:endpos,num].flatten()/expected_res[startpos:endpos,num].flatten(),label = "actual output/expected output")
    plt.plot(np.ones(endpos-startpos-1),label = "expected output/expected output")
    plt.title(f"row={row_index[num]}")
    plt.ylabel("*100%")
    plt.xlabel("row nums")
    plt.legend()
    plt.show()

In [ ]:
chip.compensation.initop("../chip_data/chip6_/")
# chip.compensation.row_offset=0
# chip.compensation.row_value=0
# chip.compensation.row_r_out=0
# 10列，两路TIA并行
row_index = [i for i in range(256)]
# 存储数据
real_res = np.zeros((256,256))
real_cond_res = np.zeros((256,256))
expected_res = np.zeros((256,256))

chip.setting.tia_map = [j for j in range(8) for i in range(2)]

for i in range(101):
    col_index = [j for j in range(i+1)]

    _,cond,r = chip.read4(crossbar=np.ones((256,256)),row_index=row_index,col_index=col_index,read_voltage=0.1,tg=5,gain=3,sub_base=True,from_row=False,split_type=4,row_type=0,col_type=0)
    cond = chip.compensation.compensation_forward(col_index,r,from_row=False,return_type=0,parallel=8)
    pos = np.ix_(row_index,col_index)
    expected_res[i,:]=np.sum(c_expected_from_col[pos],axis=1).flatten()
    real_res[i,:]=r
    real_cond_res[i,:] = cond[row_index]

In [ ]:
startpos,endpos = 0,101
for num in range(9):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(real_cond_res[startpos:endpos,num].flatten(),label = "actual output")
    plt.plot(expected_res[startpos:endpos,num].flatten(),label = "expected output")
    plt.title(f"col={col_index[num]}")
    plt.ylabel("uS")
    plt.xlabel("row nums")
    plt.legend()
    # plt.show()
    plt.subplot(1,2,2)
    plt.plot(real_cond_res[startpos:endpos,num].flatten()/expected_res[startpos:endpos,num].flatten(),label = "actual output/expected output")
    plt.plot(np.ones(endpos-startpos-1),label = "expected output/expected output")
    plt.title(f"col={col_index[num]}")
    plt.ylabel("*100%")
    plt.xlabel("row nums")
    plt.legend()
    plt.show()

# 2. 计算补偿参数

## 2.1 从行给信号列出

In [ ]:
r_wire = chip.compensation.r_wire
r_out_col = chip.compensation.col_r_out


In [ ]:
def compensation_offset_from_row(x,offset,col):
    """
        计算补偿
    """
    rows = len(x)
    ans = np.zeros(rows)
    for i in range(rows):
        sum_rw = (255-i)*r_wire if col%2==0 else 0
        r_out = r_out_col[col]
        real_r = real_res[i,col]*1e3 -r_out -sum_rw + offset
        ans[i]=1/real_r*1e6
    return ans/expected_res[:rows,col]

cnt = 31
ans = np.zeros(256)

for i in range(256):
    params, covariance = curve_fit(lambda x,offset:compensation_offset_from_row(x,offset,i),[i for i in range(cnt)], np.ones(cnt), p0=[0],bounds=([-15], [15]))
    ans[i] = params[0]

In [ ]:
np.save("../chip_data/chip6_/col_offset_parallel_16.npy",ans)
print(ans)

In [ ]:
offset_col = np.load("../chip_data/chip6_/col_offset_parallel_16.npy")
def compensation_value_from_row(x,value,col):
    """
        计算补偿
    """
    rows = len(x)
    ans = np.zeros(rows)
    for i in range(rows):
        sum_rw = (255-i)*r_wire if col%2==0 else 0
        r_out = r_out_col[col]
        real_r = real_res[i,col]*1e3 -r_out -sum_rw + offset_col[col]  - r_wire*(i**value)
        ans[i]=1/real_r*1e6
    return ans/expected_res[:rows,col]

cnt = 101
ans = np.zeros(256)

for i in range(256):
    params, covariance = curve_fit(lambda x,value:compensation_value_from_row(x,value,i),[i for i in range(cnt)], np.ones(cnt), p0=[0.5],bounds=([0], [1]))
    ans[i] = params[0]

In [ ]:
np.save("../chip_data/chip6_/col_value_parallel_16.npy",ans)
print(ans)

## 2.1 从列给信号

In [ ]:
r_wire = chip.compensation.r_wire
r_out_row = chip.compensation.row_r_out

In [ ]:
def compensation_offset_from_col(x,offset,row):
    """
        计算补偿
    """
    rows = len(x)
    ans = np.zeros(rows)
    for i in range(rows):
        sum_rw = (255-i)*r_wire if row%2==1 else 0
        r_out = r_out_row[row]
        real_r = real_res[i,row]*1e3 -r_out -sum_rw + offset
        ans[i]=1/real_r*1e6
    return ans/expected_res[:rows,row]

cnt = 31
ans = np.zeros(256)

for i in range(256):
    params, covariance = curve_fit(lambda x,offset:compensation_offset_from_col(x,offset,i),[i for i in range(cnt)], np.ones(cnt), p0=[0],bounds=([-15], [15]))
    ans[i] = params[0]

In [ ]:
# np.save("../chip_data/chip6_/row_offset.npy",ans)
np.save("../chip_data/chip6_/row_offset_parallel_8.npy",ans)
# np.save("../chip_data/chip6_/row_offset_parallel_16.npy",ans)
print(ans)

In [ ]:
# offset_row = np.load("../chip_data/chip6_/row_offset.npy")
offset_row = np.load("../chip_data/chip6_/row_offset_parallel_8.npy")
def compensation_value_from_col(x,value,row):
    """
        计算补偿
    """
    rows = len(x)
    ans = np.zeros(rows)
    for i in range(rows):
        sum_rw = (255-i)*r_wire if row%2==1 else 0
        r_out = r_out_row[row]
        real_r = real_res[i,row]*1e3 -r_out -sum_rw + offset_row[row]  - r_wire*(i**value)
        ans[i]=1/real_r*1e6
    return ans/expected_res[:rows,row]

cnt = 101
ans = np.zeros(256)

for i in range(256):
    params, covariance = curve_fit(lambda x,value:compensation_value_from_col(x,value,i),[i for i in range(cnt)], np.ones(cnt), p0=[0.5],bounds=([0], [1]))
    ans[i] = params[0]

In [ ]:
np.save("../chip_data/chip6_/row_value_parallel_8.npy",ans)
print(ans)

# 3.补偿结果

### 3.1 从行给信号列出